In [2]:
#!/usr/bin/env python3
"""
Movie Rating Prediction (IMDb Movies India)
===========================================
Regression model that predicts a movie's rating (1-10) from genre, director,
actors and basic movie info.

Pipeline
--------
1. Load + clean   : handles the Latin-1 encoding, "(2019)" years, "109 min" durations,
                    drops movies that have no rating (nothing to learn from / evaluate).
2. Explore        : rating distribution, genre / decade effects, top directors and actors.
3. Features       : genre multi-hot, director + cast "target encoding" (average rating of
                    their other movies), year, duration.  Target encoding is computed
                    out-of-fold INSIDE every training split, so a movie never sees its own
                    rating (no leakage).
4. Models         : Baseline (mean), Ridge, Random Forest, Gradient Boosting.
5. Evaluate       : 5-fold cross-validation + a final 80/20 hold-out test
                    (RMSE, MAE, R2) for two feature sets:
                      A) Brief features  = genre, director, actors (+ year, duration)
                      B) A + Votes       = adds the number of IMDb votes
                    Votes are shown separately because they are only known AFTER a movie
                    has been released and rated, so B is "easier" than a true forecast.

Usage
-----
    python movie_rating_prediction.py                        # finds the CSV next to the script
    python movie_rating_prediction.py --csv path/to/IMDb_Movies_India.csv
Works as a normal script, and when pasted into a Google Colab / Jupyter cell
(upload IMDb_Movies_India.csv to the working folder first).
"""
import argparse
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

SEED = 42
OUT_DIR = "movie_output"
ACTOR_COLS = ["Actor 1", "Actor 2", "Actor 3"]
RAW_COLS = ["Year", "Duration", "Genre", "Votes", "Director"] + ACTOR_COLS
ENC_COLS = ["director_te", "director_films", "cast_te_mean", "cast_te_max", "cast_films_mean"]

_LOG = []


def log(text=""):
    """print + remember, so the same text is saved to results_summary.txt"""
    print(text)
    _LOG.append(str(text))


# --------------------------------------------------------------------------- #
# 1. Load + clean
# --------------------------------------------------------------------------- #
def find_local_csv():
    # __file__ does not exist inside a Colab / Jupyter cell -> fall back to the working folder
    here = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    for folder in (here, os.path.join(here, "data"), os.getcwd(), os.path.join(os.getcwd(), "data")):
        for name in ("IMDb_Movies_India.csv", "IMDb Movies India.csv", "IMDb_Movies_India[2].csv"):
            p = os.path.join(folder, name)
            if os.path.isfile(p):
                return p
    return None


def load_data(csv_path=None):
    csv_path = csv_path or find_local_csv()
    if csv_path is None:
        sys.exit("Could not find IMDb_Movies_India.csv. Put it next to this script "
                 "(or in a 'data' folder) or run:  python movie_rating_prediction.py --csv <path>")
    df = None
    for enc in ("utf-8", "latin-1"):  # this dataset is Latin-1, not UTF-8
        try:
            df = pd.read_csv(csv_path, encoding=enc)
            break
        except UnicodeDecodeError:
            continue
    need = set(RAW_COLS + ["Name", "Rating"])
    if df is None or need - set(df.columns):
        sys.exit(f"Unexpected file format. Needed columns: {sorted(need)}")
    n_raw = len(df)
    df = df.drop_duplicates()
    df = df.dropna(subset=["Rating"]).reset_index(drop=True)
    log(f"Loaded {n_raw} rows from {csv_path}")
    log(f"Kept {len(df)} movies that have a rating (rows without a rating cannot be used).")
    return df


# --------------------------------------------------------------------------- #
# 2. Feature builder (sklearn transformer, leakage-safe)
# --------------------------------------------------------------------------- #
def _clean_text(s):
    s = s.astype(str).str.strip()
    return s.where(~s.isin(["", "nan", "None", "NaN"]), np.nan)


class MovieFeatures(BaseEstimator, TransformerMixin):
    """Raw movie columns -> numeric feature table.

    Director and cast names are turned into numbers with *smoothed target encoding*:
    the average rating of that person's other movies, pulled toward the global mean when
    the person has few movies.  During fit the encoding of every training movie is computed
    out-of-fold, so a movie's own rating never leaks into its own features.
    """

    def __init__(self, use_votes=False, min_genre_count=30, smoothing=5.0,
                 inner_folds=5, seed=SEED):
        self.use_votes = use_votes
        self.min_genre_count = min_genre_count
        self.smoothing = smoothing
        self.inner_folds = inner_folds
        self.seed = seed

    # ---- parsing: uses no rating information ----
    @staticmethod
    def _prepare(X):
        X = pd.DataFrame(X).reset_index(drop=True)
        d = pd.DataFrame(index=X.index)
        d["year"] = pd.to_numeric(X["Year"].astype(str).str.extract(r"(\d{4})")[0], errors="coerce")
        d["duration"] = pd.to_numeric(X["Duration"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
        d["votes"] = pd.to_numeric(X["Votes"].astype(str).str.replace(r"[^\d.]", "", regex=True),
                                   errors="coerce")
        genre = _clean_text(X["Genre"]).fillna("")
        d["genres"] = genre.map(lambda g: [x.strip() for x in g.split(",") if x.strip()])
        d["director"] = _clean_text(X["Director"])
        for c in ACTOR_COLS:
            d[c] = _clean_text(X[c])
        return d

    # ---- target encoding tables ----
    @staticmethod
    def _fit_maps(d, y):
        dm = (pd.DataFrame({"k": d["director"].to_numpy(), "y": y}).dropna(subset=["k"])
              .groupby("k")["y"].agg(["sum", "count"]))
        parts = [pd.DataFrame({"k": d[c].to_numpy(), "y": y}) for c in ACTOR_COLS]
        am = pd.concat(parts).dropna(subset=["k"]).groupby("k")["y"].agg(["sum", "count"])
        return dm, am

    def _encode(self, d, maps, gmean):
        dm, am = maps
        m = self.smoothing
        out = pd.DataFrame(index=d.index)
        s = d["director"].map(dm["sum"]).fillna(0)
        c = d["director"].map(dm["count"]).fillna(0)
        out["director_te"] = (s + m * gmean) / (c + m)
        out["director_films"] = np.log1p(c)
        te, cnt = [], []
        for col in ACTOR_COLS:
            s = d[col].map(am["sum"]).fillna(0)
            c = d[col].map(am["count"]).fillna(0)
            te.append((s + m * gmean) / (c + m))
            cnt.append(c)
        te = pd.concat(te, axis=1)
        cnt = pd.concat(cnt, axis=1)
        out["cast_te_mean"] = te.mean(axis=1)
        out["cast_te_max"] = te.max(axis=1)
        out["cast_films_mean"] = np.log1p(cnt.mean(axis=1))
        return out[ENC_COLS]

    def _assemble(self, d, enc):
        f = pd.DataFrame(index=d.index)
        f["year"] = d["year"].fillna(self.year_median_)
        f["year_missing"] = d["year"].isna().astype(int)
        f["duration"] = d["duration"].fillna(self.duration_median_)
        f["duration_missing"] = d["duration"].isna().astype(int)
        f["n_genres"] = d["genres"].map(len)
        for g in self.genres_:
            f["genre_" + g] = d["genres"].map(lambda gs, g=g: int(g in gs))
        if self.use_votes:
            f["log_votes"] = np.log1p(d["votes"].fillna(0))
        out = pd.concat([f, enc.set_axis(d.index)], axis=1)
        self.feature_names_ = list(out.columns)
        return out

    # ---- sklearn API ----
    def fit_transform(self, X, y=None, **fit_params):
        d = self._prepare(X)
        y = np.asarray(y, dtype=float)
        self.global_mean_ = float(y.mean())
        self.year_median_ = float(d["year"].median())
        self.duration_median_ = float(d["duration"].median())
        counts = pd.Series([g for gs in d["genres"] for g in gs]).value_counts()
        self.genres_ = sorted(counts[counts >= self.min_genre_count].index)
        self.maps_ = self._fit_maps(d, y)  # used for new (unseen) movies

        enc = pd.DataFrame(np.zeros((len(d), len(ENC_COLS))), columns=ENC_COLS)
        kf = KFold(self.inner_folds, shuffle=True, random_state=self.seed)
        for tr, te in kf.split(d):  # out-of-fold encoding for the training rows
            maps = self._fit_maps(d.iloc[tr], y[tr])
            enc.iloc[te] = self._encode(d.iloc[te], maps, float(y[tr].mean())).to_numpy()
        return self._assemble(d, enc)

    def fit(self, X, y=None):
        self.fit_transform(X, y)
        return self

    def transform(self, X):
        d = self._prepare(X)
        return self._assemble(d, self._encode(d, self.maps_, self.global_mean_))


# --------------------------------------------------------------------------- #
# 3. Plot style (single hue for one series, blue + orange for two; quiet grid)
# --------------------------------------------------------------------------- #
INK, INK2, SURFACE, GRID = "#0b0b0b", "#52514e", "#fcfcfb", "#e6e5e0"
BLUE, ORANGE = "#2a78d6", "#eb6834"


def style():
    plt.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": GRID, "axes.labelcolor": INK2, "text.color": INK,
        "xtick.color": INK2, "ytick.color": INK2, "axes.titlecolor": INK,
        "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlesize": 12,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
        "axes.axisbelow": True, "font.size": 10,
    })


def save(fig, out, name):
    fig.tight_layout()
    fig.savefig(os.path.join(out, name), dpi=150)
    plt.close(fig)


# --------------------------------------------------------------------------- #
# 4. Exploration
# --------------------------------------------------------------------------- #
def explore(df, out):
    d = MovieFeatures._prepare(df[RAW_COLS])
    d["Rating"] = df["Rating"].to_numpy()
    r = d["Rating"]
    log("\n=== Data overview ===")
    log(f"Rating: mean {r.mean():.2f}, median {r.median():.2f}, std {r.std():.2f}, "
        f"range {r.min():.1f}-{r.max():.1f}")
    log("Missing values among rated movies: " + ", ".join(
        f"{c} {int(n)}" for c, n in {
            "Year": d["year"].isna().sum(), "Duration": d["duration"].isna().sum(),
            "Genre": (d["genres"].map(len) == 0).sum(), "Director": d["director"].isna().sum(),
            "Actor 1": d["Actor 1"].isna().sum()}.items()))
    log("Spearman correlation with rating: " + ", ".join(
        f"{c} {d[c].corr(r, method='spearman'):+.2f}" for c in ("year", "duration", "votes")))

    # genres
    ex = d[["genres", "Rating"]].explode("genres").dropna(subset=["genres"])
    g = ex.groupby("genres")["Rating"].agg(["mean", "count"])
    g = g[g["count"] >= 100].sort_values("mean", ascending=False)
    log("\nAverage rating by genre (genres with >=100 movies):")
    log(g.round(2).head(6).to_string())
    log("   ...")
    log(g.round(2).tail(4).to_string())
    g.to_csv(os.path.join(out, "genre_summary.csv"))

    # decades
    d["decade"] = (d["year"] // 10 * 10)
    dec = d.groupby("decade")["Rating"].agg(["mean", "count"])
    dec = dec[dec["count"] >= 30]
    log("\nAverage rating by decade:")
    log(dec.round(2).to_string())

    # directors / actors
    dr = d.groupby("director")["Rating"].agg(["mean", "count"])
    dr = dr[dr["count"] >= 5].sort_values("mean", ascending=False)
    log("\nTop directors by average rating (>=5 rated movies):")
    log(dr.round(2).head(8).to_string())
    stacked = pd.concat([pd.DataFrame({"actor": d[c], "Rating": d["Rating"]}) for c in ACTOR_COLS])
    ac = stacked.dropna().groupby("actor")["Rating"].agg(["mean", "count"])
    ac = ac[ac["count"] >= 8].sort_values("mean", ascending=False)
    log("\nTop actors by average rating (>=8 rated movies):")
    log(ac.round(2).head(8).to_string())
    dr.to_csv(os.path.join(out, "director_summary.csv"))
    ac.to_csv(os.path.join(out, "actor_summary.csv"))

    # --- charts ---
    style()
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    ax.hist(r, bins=np.arange(1, 10.6, 0.5), color=BLUE, edgecolor=SURFACE, linewidth=2)
    ax.set_title("Most movies are rated between 5 and 7")
    ax.set_xlabel("IMDb rating")
    ax.set_ylabel("Number of movies")
    ax.grid(axis="x", visible=False)
    save(fig, out, "rating_distribution.png")

    top = g.head(10).iloc[::-1]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.barh(top.index, top["mean"], color=BLUE, height=0.65)
    for y_, (m, n) in enumerate(zip(top["mean"], top["count"])):
        ax.text(m + 0.03, y_, f"{m:.2f}  (n={int(n)})", va="center", fontsize=8.5, color=INK2)
    ax.set_xlim(0, top["mean"].max() + 1.2)
    ax.set_title("Highest-rated genres (genres with 100+ movies)")
    ax.set_xlabel("Average rating")
    ax.grid(axis="y", visible=False)
    save(fig, out, "avg_rating_by_genre.png")

    fig, ax = plt.subplots(figsize=(6.5, 3.4))
    ax.plot(dec.index.astype(int), dec["mean"], color=BLUE, linewidth=2, marker="o", markersize=7,
            markeredgecolor=SURFACE, markeredgewidth=2)
    ax.set_title("Average rating by decade")
    ax.set_xlabel("Decade")
    ax.set_ylabel("Average rating")
    ax.set_xticks(dec.index.astype(int))
    save(fig, out, "rating_by_decade.png")


# --------------------------------------------------------------------------- #
# 5. Models + evaluation
# --------------------------------------------------------------------------- #
FEATURE_SETS = {
    "A) Genre + director + actors (+ year, duration)": False,
    "B) A + number of votes": True,
}
GROUPS = {"Genre": ["Genre"], "Director": ["Director"], "Actors": ACTOR_COLS,
          "Year": ["Year"], "Duration": ["Duration"], "Votes": ["Votes"]}


def make_models():
    return {
        "Baseline (always predicts the mean)": DummyRegressor(),
        "Ridge": Pipeline([("sc", StandardScaler()), ("m", Ridge(alpha=10.0))]),
        "Random Forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                               n_jobs=-1, random_state=SEED),
        "Gradient Boosting": GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                        max_depth=3, subsample=0.8,
                                                        random_state=SEED),
    }


def make_pipeline(model, use_votes):
    return Pipeline([("feat", MovieFeatures(use_votes=use_votes)), ("model", model)])


def rmse(y, p):
    return float(np.sqrt(mean_squared_error(y, p)))


def cross_validate_all(X, y):
    cv = KFold(5, shuffle=True, random_state=SEED)
    scoring = {"rmse": "neg_root_mean_squared_error", "mae": "neg_mean_absolute_error", "r2": "r2"}
    rows = []
    for fs, use_votes in FEATURE_SETS.items():
        for name, model in make_models().items():
            res = cross_validate(make_pipeline(model, use_votes), X, y, cv=cv, scoring=scoring)
            rows.append({"feature_set": fs, "model": name,
                         "rmse": -res["test_rmse"].mean(), "mae": -res["test_mae"].mean(),
                         "r2": res["test_r2"].mean(), "rmse_std": res["test_rmse"].std()})
    return pd.DataFrame(rows).round(4)


def group_importance(pipe, X_test, y_test, groups, n_repeats=5, seed=SEED):
    """Increase in test RMSE when a group of input columns is shuffled (bigger = more important)."""
    rng = np.random.default_rng(seed)
    base = rmse(y_test, pipe.predict(X_test))
    out = {}
    for name, cols in groups.items():
        deltas = []
        for _ in range(n_repeats):
            Xp = X_test.copy()
            idx = rng.permutation(len(Xp))
            Xp[cols] = X_test[cols].iloc[idx].to_numpy()
            deltas.append(rmse(y_test, pipe.predict(Xp)) - base)
        out[name] = float(np.mean(deltas))
    return out


def holdout_and_plots(df, results, out):
    X, y = df[RAW_COLS], df["Rating"].to_numpy()
    idx_tr, idx_te = train_test_split(np.arange(len(df)), test_size=0.2, random_state=SEED)
    X_tr, X_te, y_tr, y_te = X.iloc[idx_tr], X.iloc[idx_te], y[idx_tr], y[idx_te]
    log(f"\n=== Hold-out test: train on {len(idx_tr)} movies, test on {len(idx_te)} unseen movies ===")

    fitted, importances, rows = {}, {}, []
    for fs, use_votes in FEATURE_SETS.items():
        sub = results[(results["feature_set"] == fs) & (~results["model"].str.startswith("Baseline"))]
        best = sub.sort_values("rmse").iloc[0]["model"]
        pipe = make_pipeline(make_models()[best], use_votes).fit(X_tr, y_tr)
        pred = np.clip(pipe.predict(X_te), 1, 10)
        base_pred = np.full(len(y_te), y_tr.mean())
        err = np.abs(pred - y_te)
        rows.append({"feature_set": fs, "best_model": best,
                     "rmse": rmse(y_te, pred), "mae": mean_absolute_error(y_te, pred),
                     "r2": r2_score(y_te, pred), "baseline_rmse": rmse(y_te, base_pred),
                     "within_0.5": 100 * (err <= 0.5).mean(), "within_1.0": 100 * (err <= 1.0).mean()})
        fitted[fs] = (pipe, pred)
        grp = {k: v for k, v in GROUPS.items() if use_votes or k != "Votes"}
        importances[fs] = group_importance(pipe, X_te, y_te, grp)
    hold = pd.DataFrame(rows).round(3)
    hold.to_csv(os.path.join(out, "holdout_results.csv"), index=False)
    for _, r in hold.iterrows():
        log(f"\n{r['feature_set']}  ->  best model: {r['best_model']}")
        log(f"  RMSE {r['rmse']:.3f} (baseline {r['baseline_rmse']:.3f})   MAE {r['mae']:.3f}   "
            f"R2 {r['r2']:.3f}")
        log(f"  Predictions within 0.5 rating points: {r['within_0.5']:.0f}%   "
            f"within 1.0 point: {r['within_1.0']:.0f}%")

    # sample predictions (feature set A = what the brief asks for)
    fsA = list(FEATURE_SETS)[0]
    sample = pd.DataFrame({"movie": df["Name"].iloc[idx_te].to_numpy(),
                           "year": df["Year"].astype(str).str.extract(r"(\d{4})")[0].iloc[idx_te].to_numpy(),
                           "actual": y_te, "predicted_A": fitted[fsA][1].round(2),
                           "predicted_B": fitted[list(FEATURE_SETS)[1]][1].round(2)})
    sample.to_csv(os.path.join(out, "test_predictions.csv"), index=False)
    log("\nExample predictions on unseen movies (A = genre/director/actors only):")
    log(sample.head(8).to_string(index=False))

    log("\nWhat drives the prediction (rise in test RMSE when that input is shuffled):")
    imp = pd.DataFrame(importances).round(3)
    imp.columns = ["A", "B"]
    log(imp.to_string())
    imp.to_csv(os.path.join(out, "factor_importance.csv"))

    # --- charts ---
    style()
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4), sharex=True, sharey=True)
    for ax, (fs, (pipe, pred)), title in zip(axes, fitted.items(),
                                             ["A) Genre, director, actors", "B) A + votes"]):
        ax.scatter(y_te, pred, s=14, color=BLUE, alpha=0.45, edgecolor="none")
        ax.plot([1, 10], [1, 10], color=INK2, linestyle="--", linewidth=1)
        ax.set_title(title)
        ax.set_xlabel("Actual rating")
        r2 = hold.loc[hold["feature_set"] == fs, "r2"].iloc[0]
        ax.text(0.04, 0.94, f"R2 = {r2:.2f}", transform=ax.transAxes, va="top", color=INK2)
    axes[0].set_ylabel("Predicted rating")
    save(fig, out, "predicted_vs_actual.png")

    names = list(GROUPS)
    order = sorted(names, key=lambda k: -max(importances[fs].get(k, 0) for fs in importances))
    order = order[::-1]
    fig, ax = plt.subplots(figsize=(7, 4))
    h = 0.36
    pos = np.arange(len(order))
    labels = list(FEATURE_SETS)
    ax.barh(pos + h / 2, [importances[labels[0]].get(k, np.nan) for k in order], height=h,
            color=BLUE, label="A) Genre, director, actors")
    ax.barh(pos - h / 2, [importances[labels[1]].get(k, np.nan) for k in order], height=h,
            color=ORANGE, label="B) A + votes")
    ax.set_yticks(pos)
    ax.set_yticklabels(order)
    ax.set_xlabel("Increase in test RMSE when shuffled (higher = matters more)")
    ax.set_title("What influences the predicted rating")
    ax.grid(axis="y", visible=False)
    ax.legend(frameon=False, loc="lower right")
    save(fig, out, "importance_by_factor.png")


# --------------------------------------------------------------------------- #
# 6. Main
# --------------------------------------------------------------------------- #
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", help="path to IMDb_Movies_India[2].csv")
    ap.add_argument("--out", default=OUT_DIR)
    args, _ = ap.parse_known_args()  # ignores extra arguments Jupyter/Colab passes in
    os.makedirs(args.out, exist_ok=True)

    df = load_data(args.csv)
    explore(df, args.out)

    X, y = df[RAW_COLS], df["Rating"]
    log("\n=== 5-fold cross-validation (lower RMSE / MAE is better, higher R2 is better) ===")
    results = cross_validate_all(X, y)
    results.to_csv(os.path.join(args.out, "cv_results.csv"), index=False)
    log(results.to_string(index=False))

    holdout_and_plots(df, results, args.out)

    with open(os.path.join(args.out, "results_summary.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(_LOG))
    log(f"\nSaved outputs to {os.path.abspath(args.out)}")


if __name__ == "__main__":
    main()

Loaded 15509 rows from /content/IMDb_Movies_India[2].csv
Kept 7919 movies that have a rating (rows without a rating cannot be used).

=== Data overview ===
Rating: mean 5.84, median 6.00, std 1.38, range 1.1-10.0
Missing values among rated movies: Year 0, Duration 2068, Genre 102, Director 5, Actor 1 125
Spearman correlation with rating: year -0.13, duration -0.02, votes +0.14

Average rating by genre (genres with >=100 movies):
             mean  count
genres                  
Documentary  7.61    143
Biography    6.82    142
History      6.80    128
Family       6.20    663
Musical      6.16    498
Drama        6.00   4924
   ...
          mean  count
genres               
Crime     5.73   1111
Thriller  5.54    876
Action    5.43   2358
Horror    4.78    280

Average rating by decade:
        mean  count
decade             
1930    5.81     66
1940    6.48    151
1950    6.63    385
1960    6.44    551
1970    6.36    826
1980    5.89   1134
1990    5.28   1132
2000    5.29   1448
2